# Executive Summary & Business Recommendations
## Enterprise Predictive Analytics Engine — E-commerce Domain

**Objective:** Synthesize findings from customer segmentation, dissatisfaction-risk
classification, and revenue forecasting into a single, decision-ready summary for
stakeholders — who are the customers, what drives dissatisfaction, and what's the
revenue outlook.

## 1. Load All Prior Outputs

In [1]:
import pandas as pd
import numpy as np

segments = pd.read_csv("../data/processed/customer_segments.csv")
model_comparison = pd.read_csv("../reports/model_comparison_results.csv")
forecast = pd.read_csv("../data/processed/revenue_forecast.csv", parse_dates=["month"])

print("Segments shape:", segments.shape)
print("Segments columns:", list(segments.columns))
print()
print("Model comparison:")
print(model_comparison)
print()
print("Forecast tail:")
print(forecast.tail(6))

Segments shape: (96096, 6)
Segments columns: ['customer_unique_id', 'recency', 'frequency', 'monetary', 'cluster', 'segment']

Model comparison:
                 Model  Accuracy  Precision  Recall  F1 Score
0  Logistic Regression     0.709      0.254   0.512      0.34
1        Random Forest     0.835      0.434   0.426      0.43

Forecast tail:
        month  predicted_revenue   lower_bound   upper_bound  actual_revenue
20 2018-09-01       1.655885e+06  1.342108e+06  1.955492e+06             NaN
21 2018-10-01       1.718746e+06  1.393638e+06  2.038839e+06             NaN
22 2018-11-01       1.783702e+06  1.477573e+06  2.072597e+06             NaN
23 2018-12-01       1.846563e+06  1.544663e+06  2.134434e+06             NaN
24 2019-01-01       1.911519e+06  1.580286e+06  2.221388e+06             NaN
25 2019-02-01       1.976475e+06  1.652503e+06  2.295782e+06             NaN


## 2. Customer Segment Profile Summary

Summarizing each segment by size and average Recency, Frequency, and Monetary
value — this becomes the "who are our customers" section of the report.

In [2]:
segment_summary = segments.groupby("segment").agg(
    customer_count=("customer_unique_id", "count"),
    avg_recency_days=("recency", "mean"),
    avg_frequency=("frequency", "mean"),
    avg_monetary=("monetary", "mean"),
).round(1)

segment_summary["pct_of_customers"] = (
    segment_summary["customer_count"] / segment_summary["customer_count"].sum() * 100
).round(1)

segment_summary = segment_summary.sort_values("avg_monetary", ascending=False)
segment_summary

,customer_count,avg_recency_days,avg_frequency,avg_monetary,pct_of_customers
segment,,,,,
High-Value Outliers,31,268.1,1.2,22330.6,0.0
Loyal Repeat Customers,2993,269.3,2.1,428.3,3.1
Lapsed / At Risk,39692,438.7,1.0,199.7,41.3
Recent One-Time Buyers,53380,178.3,1.0,198.0,55.5


## 3. Key Risk Driver Recap

Pulling the top dissatisfaction drivers from the classification model — this feeds
directly into operational recommendations (e.g. delivery SLAs).

In [3]:
best_model_row = model_comparison.loc[model_comparison["F1 Score"].idxmax()]
print("Best model:", best_model_row["Model"])
print("F1 Score:", best_model_row["F1 Score"])
print("Accuracy:", best_model_row["Accuracy"])
model_comparison

Best model: Random Forest
F1 Score: 0.43
Accuracy: 0.835


,Model,Accuracy,Precision,Recall,F1 Score
0,Logistic Regression,0.709,0.254,0.512,0.34
1,Random Forest,0.835,0.434,0.426,0.43


## 4. Revenue Outlook Recap

Next 6 months, predicted vs. historical trend, with confidence bounds.

In [4]:
forecast_future = forecast[forecast["actual_revenue"].isna()][
    ["month", "predicted_revenue", "lower_bound", "upper_bound"]
].reset_index(drop=True)

growth_pct = (
    (forecast_future["predicted_revenue"].iloc[-1] / forecast_future["predicted_revenue"].iloc[0]) - 1
) * 100

print(f"Projected growth over forecast window: {growth_pct:.1f}%")
forecast_future

Projected growth over forecast window: 19.4%


,month,predicted_revenue,lower_bound,upper_bound
0,2018-09-01,1.655885e+06,1.342108e+06,1.955492e+06
1,2018-10-01,1.718746e+06,1.393638e+06,2.038839e+06
2,2018-11-01,1.783702e+06,1.477573e+06,2.072597e+06
3,2018-12-01,1.846563e+06,1.544663e+06,2.134434e+06
4,2019-01-01,1.911519e+06,1.580286e+06,2.221388e+06
5,2019-02-01,1.976475e+06,1.652503e+06,2.295782e+06


## 5. Business Recommendations

**1. Retention is the biggest lever, not acquisition.**
96.8% of customers are either one-time buyers (55.5%, avg R$198 spent) or
lapsed/at-risk (41.3%, avg R$200 spent, 439 days since last purchase). Only
3.1% are loyal repeat customers. Converting even a small share of one-time
buyers into a second purchase (e.g. targeted follow-up offers 30-60 days
post-purchase) would have outsized impact given the size of this segment.

**2. Delivery speed, not price, drives dissatisfaction — fix operations first.**
The classification model (Random Forest, F1 0.43, 83.5% accuracy) shows
`delivery_delay_days` and `delivery_time_days` account for ~77% of predictive
power in flagging low-review-risk orders, far outweighing price, freight
cost, or product category. Recommendation: prioritize carrier SLA
tightening and proactive delay notifications over pricing/discount
strategies as the primary lever to reduce dissatisfaction.

**3. Investigate the High-Value Outliers segment (31 customers, avg R$22,330).**
This tiny segment's average spend is ~100x other segments — likely B2B or
bulk buyers. They're too small a group to move overall revenue but warrant
a dedicated account-management approach given their individual value.

**4. Revenue outlook is positive but plan around the confidence range,
not the